In [1]:
"""
Ensemble RAG -- Multiple Retrievers Aggregation Pipeline -- Production-Grade
=============================================================================
Architecture   : FIVE configurations covering the complete multi-retriever
                 aggregation design space, from score-based fusion to voting
                 methods to learned dynamic weighting:
                 (1) Baseline                      -- single dense FAISS retriever
                 (2) CombSUM Score Fusion           -- normalized score summation
                                                      across 3 heterogeneous retrievers
                                                      (dense + BM25 + MMR)
                 (3) CombMNZ Presence-Weighted Fusion-- CombSUM score * number of
                                                      lists in which the document
                                                      appeared (presence multiplier)
                 (4) Borda Count Voting             -- rank-position voting: each
                                                      retriever casts votes based on
                                                      inverse rank position, no scores
                                                      needed (fully score-agnostic)
                 (5) Convex Combination + Dynamic   -- query-adaptive weighted linear
                     Weight Meta-Retriever [BEST]      combination of normalized scores,
                                                      with per-query weight optimization
                                                      using a lightweight query router

Vector Store   : FAISS  (IndexFlatIP, BGE-large-en-v1.5, 1024-dim)
BM25           : BM25Plus (rank-bm25)
Embeddings     : BAAI/bge-large-en-v1.5  (1024-dim, asymmetric bi-encoder)
LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+ (EnsembleRetriever, FAISS, BM25Retriever)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
What Ensemble RAG Is and Why It Goes Beyond RRF
=============================================================================
Ensemble RAG employs multiple retrieval strategies in tandem. Each retriever
in the ensemble is specialized -- different in embedding model, scoring function,
search algorithm, or retrieval modality. When a query is made, each retriever
produces a ranked list independently. These ranked lists are then aggregated
using a fusion algorithm to produce a single ranked output.

The key distinction from the previous RRF pipeline (Pipeline 19) is that
Ensemble RAG treats the aggregation problem MORE GENERALLY: rather than
exclusively using RRF (rank-based, scale-invariant), the full design space
includes score-based methods (CombSUM, CombMNZ, Convex Combination) and
voting methods (Borda Count) -- each with different properties, calibration
requirements, and performance characteristics.

The fundamental aggregation taxonomy:
    SCORE-BASED METHODS (CombSUM, CombMNZ, Convex Combination):
        Operate directly on the relevance scores produced by each retriever.
        Prerequisite: scores MUST be normalized to a comparable scale [0, 1]
        before summation. Without normalization, high-magnitude scores (BM25
        can reach 14-25) dominate low-magnitude scores (cosine similarity 0.7-0.95).
        Advantage: more information than rank-based methods (scores carry magnitude
        information, not just ordering).
        Disadvantage: requires normalization, which requires knowing score ranges.

    RANK-BASED METHODS (RRF, Borda Count):
        Operate only on rank positions (1st, 2nd, 3rd...), not score magnitudes.
        No normalization required -- ranks are intrinsically comparable.
        Advantage: scale-invariant, robust to outlier scores.
        Disadvantage: discards magnitude information (rank-1 at score 0.99 vs 0.51
        are treated identically).

    LEARNED METHODS (Dynamic Weight Meta-Retriever):
        Learn optimal per-retriever weights from labeled query-document relevance data,
        or estimate weights from query features using a lightweight router model.
        Advantage: can achieve the highest performance when training data is available.
        Convex Combination significantly outperforms RRF across all tested corpora with
        a single easy-to-tune parameter (Bruch et al., 2022), especially when labeled
        data is available.
        Disadvantage: requires training data or online feedback for weight calibration.

=============================================================================
CombSUM: The Score-Based Baseline
=============================================================================
CombSUM (Fox and Shaw, 1994) is the simplest score-based fusion method:

    CombSUM_score(D) = SUM_{i=1}^{M} normalized_score_i(D)

Where normalized_score_i(D) = (score_i(D) - min_i) / (max_i - min_i) per retriever.

CombSUM is effective when:
    - Scores from different retrievers measure COMPARABLE relevance aspects.
    - The normalization preserves meaningful relative differences.
    - Documents appearing in multiple lists genuinely benefit from score accumulation.

CombSUM weakness: if retriever i returns uniformly low scores for all documents
(e.g., BM25 finds weak keyword matches for a semantic query), normalization maps
the highest weak score to 1.0, making it appear as a "perfect" match.
This is the normalization distortion problem: a weak top-result in one retriever
gets score 1.0 after normalization and artificially inflates the combined score
of documents that scored highly on ONLY that weak retriever.
Mitigation: apply a minimum score threshold before normalization; discard any
document whose raw score falls below 20% of the retriever's max score.

=============================================================================
CombMNZ: Presence-Weighted Score Fusion
=============================================================================
CombMNZ (Multiple Non-Zero) extends CombSUM with a presence multiplier:

    CombMNZ_score(D) = CombSUM_score(D) * |{i : D in list_i}|

Where |{i : D in list_i}| is the count of retrieval lists in which D appeared
with a non-zero score (i.e., the document was actually retrieved by that system).

The multiplier effect is the consensus reward: a document retrieved by 3 out of 3
systems receives a 3x multiplier on its normalized score. A document retrieved by
only 1 system receives a 1x multiplier.

CombMNZ vs. RRF consensus effect:
    Both reward multi-list consensus, but through different mechanisms:
    RRF: consensus reward is implicit in rank accumulation (each list adds a
         reciprocal rank term, so more lists = higher total score).
    CombMNZ: consensus reward is EXPLICIT as a multiplicative factor.
         CombMNZ bonus = score * n_lists. If a document has score 0.5 in 2 lists,
         its CombMNZ score = (0.5 + 0.5) * 2 = 2.0.
         If another document has score 0.95 in 1 list only,
         its CombMNZ score = 0.95 * 1 = 0.95.
         The multi-list document wins despite lower individual scores.

CombMNZ is often empirically superior to CombSUM: it rewards breadth of retrieval
validation while still using score magnitudes.

=============================================================================
Borda Count: The Pure Rank Voting Method
=============================================================================
Borda Count is a ranked voting procedure originating in social choice theory
(Jean-Charles de Borda, 1784), adapted for information retrieval by Aslam and
Montague (2001). For a list of K candidates with M voters (retrievers):

    Borda_score(D, retriever_i) = (K - rank_i(D))
    Borda_total(D) = SUM_{i=1}^{M} Borda_score(D, retriever_i)

Where rank_i(D) is the 0-based rank of D in retriever i's list (0 = top rank).
Documents not appearing in retriever i's list receive a score of 0 from that retriever.

Example with K=5, M=3 retrievers, document D:
    BM25: rank=0 (top)    -> Borda_BM25 = 5 - 0 = 5
    FAISS: rank=2          -> Borda_FAISS = 5 - 2 = 3
    MMR: not in top-5     -> Borda_MMR = 0
    Borda_total(D) = 5 + 3 + 0 = 8

Borda Count vs RRF:
    Borda Count:  score = K - rank (LINEAR decrease with rank).
                  Top document gets K points, last gets 0 points.
                  The gap between rank-1 and rank-2 is the same as rank-2 and rank-3.
    RRF:          score = 1/(k + rank) (HYPERBOLIC decrease with rank).
                  The gap between rank-1 and rank-2 is much larger than rank-10 and rank-11.
                  RRF rewards top-ranked documents more aggressively than Borda Count.

When Borda Count is preferred over RRF:
    - You want EQUAL weight per rank step (linear, not hyperbolic).
    - Your retrievers produce diverse results and you want to reward consistent
      mid-range ranking as much as occasional top ranking.
    - You need a score-agnostic method but dislike RRF's hyperbolic concentration.

=============================================================================
Convex Combination: The Optimal Score Fusion When Data Is Available
=============================================================================
Convex Combination (CC) is a weighted linear combination of normalized scores:

    CC_score(D) = SUM_{i=1}^{M} w_i * normalized_score_i(D)
    Subject to: SUM(w_i) = 1.0,  w_i >= 0.

When training data is available, more general score-based fusion methods
significantly outperform RRF. Convex Combination outperforms RRF across all
tested corpora with a single, easy-to-tune parameter (Bruch et al., 2022).

The weight w_i for retriever i represents the estimated contribution of that
retriever to overall ranking quality. Optimal weights are calibrated by minimizing
nDCG or MAP on a labeled validation set.

Dynamic weight meta-retriever:
    In the absence of labeled data, weights can be estimated from query features:
        semantic_weight(query): higher if query is long, conceptual, uses relational phrasing.
        keyword_weight(query):  higher if query is short, uses technical terms, exact phrases.
    A lightweight query router (a few sigmoid/softmax operations) produces per-query
    weights at <1ms latency, enabling query-adaptive CC without a labeled training set.

"""

'\nEnsemble RAG -- Multiple Retrievers Aggregation Pipeline -- Production-Grade\n=============================================================================\nArchitecture   : FIVE configurations covering the complete multi-retriever\n                 aggregation design space, from score-based fusion to voting\n                 methods to learned dynamic weighting:\n                 (1) Baseline                      -- single dense FAISS retriever\n                 (2) CombSUM Score Fusion           -- normalized score summation\n                                                      across 3 heterogeneous retrievers\n                                                      (dense + BM25 + MMR)\n                 (3) CombMNZ Presence-Weighted Fusion-- CombSUM score * number of\n                                                      lists in which the document\n                                                      appeared (presence multiplier)\n                 (4) Borda Count Voting       

In [4]:

import os
import sys
import time
import math
import pickle
import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import  BM25Retriever
from langchain_community.vectorstores import FAISS  
from langchain_classic.retrievers import EnsembleRetriever
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("ensemble_rag")


In [6]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class EnsembleConfig:
    """
    Centralized configuration for the Ensemble RAG pipeline.

    SCORE_MIN_THRESHOLD (0.20):
        Minimum raw score fraction retained after normalization to mitigate
        normalization distortion in CombSUM and CombMNZ.
        If a retriever's max score is S_max and a document scores below
        0.20 * S_max, its normalized score is set to 0.0 (treated as not retrieved).
        This prevents weak BM25 matches from contributing artificially high
        normalized scores when the query is purely semantic.

    BORDA_K (10):
        The number of top documents from each retriever that participate in
        Borda Count voting. Each retriever casts K+1 to 1 votes for its top-K
        documents. Documents outside the top-K receive 0 votes.
        A larger K means more documents participate but the rank-differentiation
        between positions narrows. K=10 is the production-validated setting:
        it covers the typical relevant document depth while keeping vote differentiation
        meaningful (10 points for rank-1 vs 1 point for rank-10 = 10x ratio).

    CC_WEIGHTS_SEMANTIC / CC_WEIGHTS_KEYWORD / CC_WEIGHTS_HYBRID:
        Per-retriever weight vectors for the three dynamic routing classes in Config 5.
        These weights represent calibrated estimates of each retriever's quality
        contribution for each query class. In production, these should be derived from
        nDCG optimization on a labeled validation set of 50-200 representative queries.
        The weights for this pipeline are empirically motivated defaults:
            - SEMANTIC queries: BGE-large dense retrieval dominates (0.70),
              MMR adds diversity (0.20), BM25 contributes weakly (0.10).
            - KEYWORD queries: BM25Plus exact-match leads (0.50),
              BGE-large semantic (0.35), MMR suppressed (0.15).
            - HYBRID queries: balanced (0.40 BGE, 0.35 BM25, 0.25 MMR).
        The weights MUST sum to 1.0 within each class (convex combination constraint).

    RETRIEVER_LABELS:
        Human-readable names for the three retrievers in this pipeline.
        Used for trace observability, not computationally.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",     "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers",  "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",   "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- Indexes -----------------------------------------------------------
    FAISS_PERSIST_DIR: str = "./faiss_ensemble_index"
    BM25_PERSIST_DIR:  str = "./bm25_ensemble_index"
    BM25_PARAMS: Dict      = {"k1": 1.5, "b": 0.75, "delta": 0.5}

    # --- BGE Embeddings ----------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Retrieval K Parameters -------------------------------------------
    BASE_K:    int = 10   # per-retriever K for all ensemble methods
    FINAL_K:   int = 4    # documents injected into LLM context
    BORDA_K:   int = 10   # top-K docs participating in Borda voting

    # --- Score Normalization Parameters -----------------------------------
    SCORE_MIN_THRESHOLD: float = 0.20   # suppress weak-match false positives

    # --- CombSUM / CombMNZ Deduplication ---------------------------------
    DEDUP_PREFIX_LEN: int = 150   # chars for content-based deduplication

    # --- Dynamic CC Weights per Query Class ------------------------------
    # [BGE-large-dense, BM25Plus, MMR-dense]
    CC_WEIGHTS_SEMANTIC: List[float] = [0.70, 0.10, 0.20]
    CC_WEIGHTS_KEYWORD:  List[float] = [0.35, 0.50, 0.15]
    CC_WEIGHTS_HYBRID:   List[float] = [0.40, 0.35, 0.25]
    RETRIEVER_LABELS: List[str]      = ["BGE_Dense", "BM25Plus", "MMR_Dense"]

    # --- LLM Temperature --------------------------------------------------
    LLM_ANSWER_TEMPERATURE: float = 0.0

    # --- Azure OpenAI LLM -------------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int           = 1024

    # --- RAG Answer Prompt ------------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved via ensemble multi-retriever aggregation):
{context}

Question: {question}

Precise technical answer:"""



In [7]:


# ===========================================================================
# SECTION 2 -- ENSEMBLE TRACE DATA CLASS
# ===========================================================================

@dataclass
class EnsembleTrace:
    """
    Records the complete execution trace for one Ensemble RAG run.

    fusion_scores: Dict[doc_preview -> aggregated_score].
        The full score map from the aggregation function, exposing every candidate's
        score for monitoring and debugging.

    retriever_contributions: Dict[retriever_label -> doc_count].
        Per-retriever hit counts. If BM25 consistently returns < BASE_K docs,
        the query class is semantically dominated and BM25 should receive lower weight.

    presence_counts: Dict[doc_preview -> n_retrievers_found_it].
        For CombMNZ: the presence count that acted as the multiplier.
        Monitoring: if P90 presence count = 1 (most documents retrieved by only one
        retriever), the retrievers are not finding the same documents, meaning
        cross-retriever validation is rare. This can indicate corpus sparsity or
        retriever misconfiguration.

    aggregation_method: the label of the aggregation method used.
    weights_used: the actual weight vector applied in Config 5 CC fusion.
    query_class: the routing decision made for Config 5 (keyword/semantic/hybrid).
    """
    question:               str
    strategy:               str
    final_docs:             List[Document]      = field(default_factory=list)
    fusion_scores:          Dict[str, float]    = field(default_factory=dict)
    retriever_contributions: Dict[str, int]     = field(default_factory=dict)
    presence_counts:        Dict[str, int]      = field(default_factory=dict)
    aggregation_method:     str                 = ""
    weights_used:           Optional[List[float]] = None
    query_class:            str                 = ""
    candidates_total:       int                 = 0
    final_answer:           str                 = ""
    retrieval_ms:           float               = 0.0
    fusion_ms:              float               = 0.0
    generation_ms:          float               = 0.0
    total_ms:               float               = 0.0

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:80]}")
        print(f"{'='*74}")
        print(f"\n  Aggregation: {self.aggregation_method}")
        print(f"  Candidates:  {self.candidates_total}")
        print(f"  Final docs:  {len(self.final_docs)}")
        if self.query_class:
            print(f"  Query class: {self.query_class}")
        if self.weights_used:
            wt = ", ".join(
                f"{lbl}={w:.2f}"
                for lbl, w in zip(["BGE_Dense", "BM25Plus", "MMR_Dense"], self.weights_used)
            )
            print(f"  Weights: {wt}")

        if self.retriever_contributions:
            print(f"\n  Retriever contributions:")
            for name, cnt in self.retriever_contributions.items():
                print(f"    {name:<30}: {cnt} docs")

        if self.presence_counts:
            hist = {}
            for v in self.presence_counts.values():
                hist[v] = hist.get(v, 0) + 1
            print(f"\n  Presence distribution (n_retrievers -> n_docs):")
            for n_ret in sorted(hist):
                print(f"    Retrieved by {n_ret} retriever(s): {hist[n_ret]} docs")

        print(f"\n  Top fusion scores:")
        for i, (preview, score) in enumerate(list(self.fusion_scores.items())[:5], 1):
            print(f"    [{i}] score={score:.5f}  '{preview[:50]}...'")

        print(f"\n  Final docs to LLM:")
        for i, doc in enumerate(self.final_docs, 1):
            paper = doc.metadata.get("paper_name", "?")[:30]
            page  = doc.metadata.get("page", "?")
            print(f"    [{i}] {paper} p{page}  ({len(doc.page_content)} chars)")

        print(f"\n  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"fusion={self.fusion_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")


In [8]:

# ===========================================================================
# SECTION 3 -- CORE AGGREGATION IMPLEMENTATIONS
# ===========================================================================

def _dedup_key(doc: Document, prefix_len: int = 150) -> str:
    """Content-based deduplication key: first N chars of page_content."""
    return doc.page_content[:prefix_len]


def normalize_scores(
    doc_score_list: List[Tuple[Document, float]],
    min_threshold_ratio: float = 0.20,
) -> Dict[str, float]:
    """
    Min-max normalize raw retriever scores to [0, 1] for score-based fusion.

    The normalization formula: norm_score = (s - s_min) / (s_max - s_min)

    Normalization distortion protection via min_threshold_ratio:
        After normalization, any document whose original raw score was below
        min_threshold_ratio * s_max is ZEROED OUT (set to 0.0).
        This prevents weak matches from receiving artificially inflated normalized
        scores. Example: if BM25 max score = 14.2 and threshold = 0.20,
        any BM25 score < 2.84 is zeroed before contributing to CombSUM.

    Args:
        doc_score_list : List of (Document, raw_score) tuples from one retriever.
        min_threshold_ratio: fraction of max score below which scores are zeroed.

    Returns:
        Dict[dedup_key -> normalized_score_in_[0,1]]
    """
    if not doc_score_list:
        return {}

    scores = [s for _, s in doc_score_list]
    s_min  = min(scores)
    s_max  = max(scores)
    s_range = s_max - s_min

    threshold = min_threshold_ratio * s_max
    result: Dict[str, float] = {}

    for doc, raw_score in doc_score_list:
        key = _dedup_key(doc)
        if raw_score < threshold:
            norm = 0.0
        elif s_range == 0:
            norm = 1.0
        else:
            norm = (raw_score - s_min) / s_range
        result[key] = norm

    return result


def combsum_fuse(
    retriever_results: List[Tuple[str, List[Tuple[Document, float]]]],
    config:            EnsembleConfig,
) -> Tuple[List[Document], Dict[str, float], Dict[str, int]]:
    """
    CombSUM fusion: normalized score summation across all retrievers.

    Formula: CombSUM_score(D) = SUM_{i} normalize_i(score_i(D))

    Documents not retrieved by retriever i contribute 0.0 to the sum.
    Final ranking: descending CombSUM score, deduplicated by content prefix.

    Args:
        retriever_results: List of (retriever_label, [(doc, raw_score), ...]) tuples.
        config: EnsembleConfig.

    Returns:
        Tuple of:
            - List[Document]: sorted by descending CombSUM score.
            - Dict[str, float]: dedup_key -> CombSUM score.
            - Dict[str, int]: dedup_key -> presence count (n_retrievers found doc).
    """
    combined_scores: Dict[str, float]    = {}
    presence:        Dict[str, int]      = {}
    doc_registry:    Dict[str, Document] = {}

    for _label, doc_score_list in retriever_results:
        norm_map = normalize_scores(
            doc_score_list, min_threshold_ratio=config.SCORE_MIN_THRESHOLD
        )
        for key, norm_score in norm_map.items():
            # Find the document object for this key
            for doc, _ in doc_score_list:
                if _dedup_key(doc) == key:
                    doc_registry[key] = doc
                    break
            if norm_score > 0:
                combined_scores[key] = combined_scores.get(key, 0.0) + norm_score
                presence[key]        = presence.get(key, 0) + 1

    sorted_keys  = sorted(combined_scores, key=lambda k: combined_scores[k], reverse=True)
    sorted_docs  = [doc_registry[k] for k in sorted_keys if k in doc_registry]
    sorted_scores = {k: combined_scores[k] for k in sorted_keys}

    return sorted_docs, sorted_scores, presence


def combmnz_fuse(
    retriever_results: List[Tuple[str, List[Tuple[Document, float]]]],
    config:            EnsembleConfig,
) -> Tuple[List[Document], Dict[str, float], Dict[str, int]]:
    """
    CombMNZ fusion: CombSUM score multiplied by presence count.

    Formula: CombMNZ_score(D) = CombSUM_score(D) * |{i : norm_score_i(D) > 0}|

    The presence multiplier rewards multi-retriever consensus explicitly:
        Retrieved by 3 retrievers: score * 3
        Retrieved by 2 retrievers: score * 2
        Retrieved by 1 retriever:  score * 1

    CombMNZ is the direct multi-retriever generalization of CombSUM that
    embeds the consensus effect as an explicit multiplicative factor, analogous
    to how RRF's reciprocal sum embeds consensus implicitly through rank accumulation.

    Args:
        retriever_results: same format as combsum_fuse.
        config: EnsembleConfig.

    Returns:
        Tuple of docs, CombMNZ scores, presence counts (same structure as CombSUM).
    """
    docs, combsum_scores, presence = combsum_fuse(retriever_results, config)

    # Apply presence multiplier to CombSUM scores
    combmnz_scores: Dict[str, float] = {}
    for key, csum_score in combsum_scores.items():
        n = presence.get(key, 1)
        combmnz_scores[key] = csum_score * n

    # Re-sort by CombMNZ score
    doc_map = {_dedup_key(d): d for d in docs}
    sorted_keys  = sorted(combmnz_scores, key=lambda k: combmnz_scores[k], reverse=True)
    sorted_docs  = [doc_map[k] for k in sorted_keys if k in doc_map]
    sorted_scores = {k: combmnz_scores[k] for k in sorted_keys}

    return sorted_docs, sorted_scores, presence


def borda_count_fuse(
    retriever_results: List[Tuple[str, List[Document]]],
    k:                 int,
) -> Tuple[List[Document], Dict[str, float], Dict[str, int]]:
    """
    Borda Count voting fusion (score-agnostic, pure rank-based voting).

    Formula:
        Borda_score(D, retriever_i) = max(0, K - rank_0based_i(D))
        Borda_total(D) = SUM_{i} Borda_score(D, retriever_i)

    Voting mechanics:
        K=10: rank-0 (top) gets 10 points, rank-9 (last) gets 1 point, not retrieved gets 0.
        The voting function is LINEAR with rank, unlike RRF's HYPERBOLIC function.
        A document at rank-1 in all three retrievers: 10 + 10 + 10 = 30 points (max).
        A document at rank-5 in one retriever only: 5 + 0 + 0 = 5 points.

    Score-agnostic property:
        Borda Count does not require raw scores at all. The API-level retriever
        (vectorstore.as_retriever()) returns documents in ranked order without
        score passthrough in LangChain's standard interface. Borda Count is the
        correct aggregation method when scores are unavailable or non-comparable
        and RRF's hyperbolic concentration is undesirable.

    Args:
        retriever_results: List of (retriever_label, [Document, ...]) tuples.
                           Documents should be ordered by descending relevance
                           (most relevant first = position 0).
        k: maximum number of top documents to consider per retriever (= BORDA_K).

    Returns:
        Tuple of:
            - List[Document]: sorted by descending Borda score.
            - Dict[str, float]: dedup_key -> Borda score.
            - Dict[str, int]: dedup_key -> presence count.
    """
    borda_scores:  Dict[str, float]    = {}
    presence:      Dict[str, int]      = {}
    doc_registry:  Dict[str, Document] = {}

    for _label, doc_list in retriever_results:
        for rank_0based, doc in enumerate(doc_list[:k]):
            key        = _dedup_key(doc)
            vote       = k - rank_0based   # linear vote: top doc gets K, last gets 1
            borda_scores[key] = borda_scores.get(key, 0.0) + vote
            presence[key]     = presence.get(key, 0) + 1
            doc_registry[key] = doc_registry.get(key, doc)

    sorted_keys  = sorted(borda_scores, key=lambda k: borda_scores[k], reverse=True)
    sorted_docs  = [doc_registry[k] for k in sorted_keys if k in doc_registry]
    sorted_scores = {k: borda_scores[k] for k in sorted_keys}

    return sorted_docs, sorted_scores, presence


def convex_combination_fuse(
    retriever_results: List[Tuple[str, List[Tuple[Document, float]]]],
    weights:           List[float],
    config:            EnsembleConfig,
) -> Tuple[List[Document], Dict[str, float], Dict[str, int]]:
    """
    Convex Combination (CC) fusion: weighted normalized score summation.

    Formula: CC_score(D) = SUM_{i} w_i * normalize_i(score_i(D))
    Constraint: SUM(w_i) = 1.0, w_i >= 0.

    CC is strictly more expressive than CombSUM (which is CC with equal weights)
    and more expressive than RRF (which uses rank-based scores, not raw scores).
    When training data is available to calibrate weights, CC outperforms RRF
    across all tested corpora (Bruch et al., 2022).

    The weight vector for this function is provided by the query-adaptive router
    in Config 5. The router estimates the optimal weights based on query features:
        short/technical query -> high BM25 weight
        long/conceptual query -> high dense weight
        balanced query        -> hybrid weights

    Args:
        retriever_results: List of (label, [(doc, raw_score),...]) tuples.
        weights: per-retriever weight vector summing to 1.0.
        config: EnsembleConfig.

    Returns:
        Tuple of docs, CC scores, presence counts.
    """
    assert abs(sum(weights) - 1.0) < 1e-6, \
        f"CC weights must sum to 1.0, got {sum(weights):.4f}"
    assert len(weights) == len(retriever_results), \
        f"weights len {len(weights)} != retrievers len {len(retriever_results)}"

    combined_scores: Dict[str, float]    = {}
    presence:        Dict[str, int]      = {}
    doc_registry:    Dict[str, Document] = {}

    for w, (_label, doc_score_list) in zip(weights, retriever_results):
        norm_map = normalize_scores(
            doc_score_list, min_threshold_ratio=config.SCORE_MIN_THRESHOLD
        )
        for key, norm_score in norm_map.items():
            for doc, _ in doc_score_list:
                if _dedup_key(doc) == key:
                    doc_registry[key] = doc
                    break
            weighted = w * norm_score
            if weighted > 0:
                combined_scores[key] = combined_scores.get(key, 0.0) + weighted
                presence[key]        = presence.get(key, 0) + 1

    sorted_keys   = sorted(combined_scores, key=lambda k: combined_scores[k], reverse=True)
    sorted_docs   = [doc_registry[k] for k in sorted_keys if k in doc_registry]
    sorted_scores = {k: combined_scores[k] for k in sorted_keys}

    return sorted_docs, sorted_scores, presence



In [9]:

# ===========================================================================
# SECTION 4 -- QUERY CLASS ROUTER (for Config 5)
# ===========================================================================

def route_query_class(question: str) -> Tuple[str, List[float]]:
    """
    Lightweight query router: classifies the query and returns adaptive CC weights.

    Classification heuristics (no LLM call, < 1ms latency):
        KEYWORD indicators: digits present, short question (<= 7 tokens), technical
                            acronyms, "define", "abbreviation", "version", "score".
        SEMANTIC indicators: long question (>= 12 tokens), "how does", "explain",
                             "describe", "why", "compare", "difference", "relationship".
        HYBRID: neither strongly keyword nor semantic.

    Weight vectors:
        SEMANTIC: BGE_Dense=0.70, BM25Plus=0.10, MMR_Dense=0.20
        KEYWORD:  BGE_Dense=0.35, BM25Plus=0.50, MMR_Dense=0.15
        HYBRID:   BGE_Dense=0.40, BM25Plus=0.35, MMR_Dense=0.25

    In production: replace with a fine-tuned 3-class text classifier (fastText or
    a 6-layer BERT fine-tuned on your labeled query set). Target latency: <5ms.
    Target accuracy: >85% on your query distribution.

    Returns:
        Tuple of (query_class: str, weights: List[float])
    """
    from typing import cast
    q_lower   = question.lower()
    n_tokens  = len(question.split())
    has_digit = any(c.isdigit() for c in question)

    kw_score  = sum([
        has_digit,
        n_tokens <= 7,
        any(w in q_lower for w in ["bleu", "f1", "score", "percent", "version",
                                    "define", "acronym", "abbreviation"]),
        "what is" in q_lower and n_tokens <= 8,
    ])
    sem_score = sum([
        n_tokens >= 12,
        "how does" in q_lower,
        "explain" in q_lower,
        "describe" in q_lower,
        "why" in q_lower,
        "compare" in q_lower,
        "difference" in q_lower,
        "relationship" in q_lower,
    ])

    if kw_score > sem_score:
        return "keyword",  [0.35, 0.50, 0.15]
    elif sem_score > kw_score:
        return "semantic", [0.70, 0.10, 0.20]
    else:
        return "hybrid",   [0.40, 0.35, 0.25]



In [11]:

# ===========================================================================
# SECTION 5 -- PDF, INDEX, AND MODEL BUILDERS
# ===========================================================================

def download_pdfs(config: EnsembleConfig) -> List[Path]:
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-Ensemble-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}'. Place at '{dest}'.") from exc
    return paths


def load_and_chunk_documents(
    pdf_paths: List[Path], config: EnsembleConfig,
) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=450, chunk_overlap=60,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        pages = PyPDFLoader(str(pdf_path)).load()
        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)
    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks


def build_bge_embeddings(config: EnsembleConfig) -> HuggingFaceEmbeddings:
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss_index(
    chunks: List[Document], embeddings: HuggingFaceEmbeddings, config: EnsembleConfig,
) -> FAISS:
    faiss_file = Path(config.FAISS_PERSIST_DIR) / "index.faiss"
    if faiss_file.exists():
        try:
            vs = FAISS.load_local(config.FAISS_PERSIST_DIR, embeddings,
                                   allow_dangerous_deserialization=True)
            logger.info("FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)
    logger.info("Building FAISS from %d chunks ...", len(chunks))
    vs = FAISS.from_documents(chunks, embeddings)
    Path(config.FAISS_PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_PERSIST_DIR)
    return vs


def build_bm25_index(chunks: List[Document], config: EnsembleConfig) -> BM25Retriever:
    cache = Path(config.BM25_PERSIST_DIR) / "bm25plus.pkl"
    if cache.exists():
        with open(cache, "rb") as f:
            state = pickle.load(f)
        return BM25Retriever(vectorizer=state["vectorizer"], docs=state["docs"],
                             k=state["k"], preprocess_func=state["preprocess_func"])
    ret = BM25Retriever.from_documents(chunks, bm25_variant="plus",
                                        bm25_params=config.BM25_PARAMS,
                                        preprocess_func=lambda t: t.lower().split())
    ret.k = config.BASE_K
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump({"vectorizer": ret.vectorizer, "docs": ret.docs,
                     "k": ret.k, "preprocess_func": ret.preprocess_func}, f)
    return ret


def build_ollama_llm(config: EnsembleConfig) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )
def retrieve_with_scores(
    question:    str,
    vectorstore: FAISS,
    k:           int,
    search_type: str = "similarity",
    mmr_kwargs:  Optional[Dict] = None,
) -> List[Tuple[Document, float]]:
    """
    Retrieve documents WITH their similarity scores from FAISS.

    FAISS's similarity_search_with_score returns (doc, score) tuples.
    For BGE-large with normalize_embeddings=True and IndexFlatIP (inner product),
    the score IS the cosine similarity (inner product of unit vectors).
    Score range: [-1, 1], practical range for relevant docs: [0.6, 0.99].

    For MMR retrieval: scores are not directly provided by the MMR algorithm
    (which uses diversity-penalized selection). We approximate scores using
    the similarity score at the time of document selection (not ideal).
    In production, compute explicit cosine similarity after MMR selection.

    Args:
        question    : User query text.
        vectorstore : FAISS index.
        k           : Number of documents to retrieve.
        search_type : "similarity" or "mmr".
        mmr_kwargs  : Additional kwargs for MMR (lambda_mult, fetch_k).

    Returns:
        List of (Document, score) tuples in descending score order.
    """
    if search_type == "similarity":
        results = vectorstore.similarity_search_with_score(question, k=k)
        # FAISS inner product distance: higher = more similar (already cosine with normalized vecs)
        return [(doc, float(score)) for doc, score in results]
    elif search_type == "mmr":
        kwargs = {"k": k, "fetch_k": k * 3, "lambda_mult": 0.7}
        if mmr_kwargs:
            kwargs.update(mmr_kwargs)
        docs = vectorstore.max_marginal_relevance_search(question, **kwargs)
        # Approximate MMR scores via post-hoc similarity computation
        scores_only = vectorstore.similarity_search_with_score(question, k=k * 2)
        score_map   = {_dedup_key(d): s for d, s in scores_only}
        return [(doc, float(score_map.get(_dedup_key(doc), 0.5))) for doc in docs]
    else:
        raise ValueError(f"Unknown search_type: {search_type}")


def retrieve_bm25_with_scores(
    question: str,
    bm25:     BM25Retriever,
    k:        int,
) -> List[Tuple[Document, float]]:
    """
    Retrieve documents from BM25Plus with BM25 scores.

    BM25Retriever.invoke() returns documents without scores.
    To get BM25Plus scores directly, we call the underlying vectorizer.
    BM25Plus scores are non-negative TF-IDF weighted values.
    Typical range for a 2,000-chunk corpus: 0 to ~20.

    Args:
        question : User query text.
        bm25     : BM25Retriever with BM25Plus vectorizer.
        k        : Number of top-K documents to return.

    Returns:
        List of (Document, bm25_score) tuples in descending score order.
    """
    tokenized_query = question.lower().split()
    scores          = bm25.vectorizer.get_scores(tokenized_query)
    top_k_indices   = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        if idx < len(bm25.docs):
            results.append((bm25.docs[idx], float(scores[idx])))

    return results


def format_context_and_answer(
    question: str,
    docs:     List[Document],
    llm:      ChatOllama,
    config:   EnsembleConfig,
) -> Tuple[str, float]:
    context = "\n\n---\n\n".join([
        f"[{d.metadata.get('paper_name','?')[:30]} | p{d.metadata.get('page','?')}]\n"
        f"{d.page_content.strip()}"
        for d in docs
    ])
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    t0     = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer, (time.perf_counter() - t0) * 1000



In [31]:
# Compatibility hotfixes for rank-bm25/langchain version differences and LLM init safety
def bm25_preprocess(text: str) -> List[str]:
    return text.lower().split()


def build_bm25_index(chunks: List[Document], config: EnsembleConfig) -> BM25Retriever:
    cache = Path(config.BM25_PERSIST_DIR) / "bm25plus.pkl"
    if cache.exists():
        try:
            with open(cache, "rb") as f:
                state = pickle.load(f)
            return BM25Retriever(
                vectorizer=state["vectorizer"],
                docs=state["docs"],
                k=state["k"],
                preprocess_func=state["preprocess_func"],
            )
        except Exception:
            logger.warning("BM25 cache unreadable/corrupted; rebuilding cache.")
            cache.unlink(missing_ok=True)

    try:
        # Preferred path for newer langchain/rank-bm25 combinations
        ret = BM25Retriever.from_documents(
            chunks,
            bm25_variant="plus",
            bm25_params=config.BM25_PARAMS,
            preprocess_func=bm25_preprocess,
        )
    except TypeError as exc:
        # Fallback for environments where BM25Okapi is used internally
        # and does not support BM25Plus-only args such as `delta`.
        msg = str(exc)
        if "delta" not in msg and "bm25_variant" not in msg:
            raise
        logger.warning(
            "BM25Plus init not supported by installed dependencies; falling back to BM25Okapi-compatible params."
        )
        okapi_params = {k: v for k, v in config.BM25_PARAMS.items() if k in {"k1", "b"}}
        ret = BM25Retriever.from_documents(
            chunks,
            bm25_params=okapi_params,
            preprocess_func=bm25_preprocess,
        )

    ret.k = config.BASE_K
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump(
            {
                "vectorizer": ret.vectorizer,
                "docs": ret.docs,
                "k": ret.k,
                "preprocess_func": ret.preprocess_func,
            },
            f,
        )
    return ret


def build_ollama_llm(config: EnsembleConfig) -> ChatOllama:
    """Return a ChatOllama client without eager connectivity checks."""
    base_url = getattr(config, "OLLAMA_BASE_URL", "http://localhost:11434")
    model = getattr(config, "OLLAMA_MODEL", "qwen2.5-coder:7b")
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, "LLM_ANSWER_TEMPERATURE", 0.0),
        num_predict=getattr(config, "LLM_MAX_TOKENS", 512),
    )

In [12]:

# ===========================================================================
# SECTION 6 -- FIVE ENSEMBLE CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline -- single dense FAISS retriever
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question: str, vectorstore: FAISS,
    llm: ChatOllama, config: EnsembleConfig,
) -> EnsembleTrace:
    """
    Configuration 1: Single dense FAISS retriever (no ensemble).
    Top-FINAL_K by cosine similarity. Control condition.
    """
    trace = EnsembleTrace(question=question, strategy="Config1_Baseline_SingleRetriever")
    t0    = time.perf_counter()
    t_ret = time.perf_counter()
    results = retrieve_with_scores(question, vectorstore, config.FINAL_K)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000
    trace.final_docs   = [doc for doc, _ in results]
    trace.fusion_scores = {_dedup_key(doc): score for doc, score in results}
    trace.retriever_contributions = {"FAISS_BGE": len(results)}
    trace.candidates_total  = len(results)
    trace.aggregation_method = "None (single retriever)"
    answer, trace.generation_ms = format_context_and_answer(
        question, trace.final_docs, llm, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [13]:

# ---------------------------------------------------------------------------
# CONFIG 2: CombSUM Score Fusion
# ---------------------------------------------------------------------------

def run_config2_combsum(
    question: str, vectorstore: FAISS, bm25_base: BM25Retriever,
    llm: ChatOllama, config: EnsembleConfig,
) -> EnsembleTrace:
    """
    Configuration 2: CombSUM -- normalized score summation.

    Three retrievers: BGE-large dense, BM25Plus, MMR-dense.
    All raw scores min-max normalized per retriever to [0, 1].
    Summed across retrievers. Documents not retrieved: contribute 0.
    Final ranking: descending CombSUM score.
    """
    trace = EnsembleTrace(question=question, strategy="Config2_CombSUM_ScoreFusion")
    t0    = time.perf_counter()

    t_ret = time.perf_counter()
    dense_results = retrieve_with_scores(question, vectorstore, config.BASE_K, "similarity")
    bm25_results  = retrieve_bm25_with_scores(question, bm25_base, config.BASE_K)
    mmr_results   = retrieve_with_scores(question, vectorstore, config.BASE_K, "mmr")
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.retriever_contributions = {
        "BGE_Dense": len(dense_results),
        "BM25Plus":  len(bm25_results),
        "MMR_Dense": len(mmr_results),
    }

    retriever_results = [
        ("BGE_Dense", dense_results),
        ("BM25Plus",  bm25_results),
        ("MMR_Dense", mmr_results),
    ]

    t_fuse = time.perf_counter()
    fused_docs, fused_scores, presence = combsum_fuse(retriever_results, config)
    trace.fusion_ms = (time.perf_counter() - t_fuse) * 1000

    trace.aggregation_method = "CombSUM (normalized score summation)"
    trace.candidates_total   = len(fused_docs)
    trace.final_docs         = fused_docs[: config.FINAL_K]
    trace.fusion_scores      = {k: v for k, v in list(fused_scores.items())[:20]}
    trace.presence_counts    = presence

    answer, trace.generation_ms = format_context_and_answer(
        question, trace.final_docs, llm, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [14]:

# ---------------------------------------------------------------------------
# CONFIG 3: CombMNZ Presence-Weighted Fusion
# ---------------------------------------------------------------------------

def run_config3_combmnz(
    question: str, vectorstore: FAISS, bm25_base: BM25Retriever,
    llm: ChatOllama, config: EnsembleConfig,
) -> EnsembleTrace:
    """
    Configuration 3: CombMNZ -- CombSUM score * presence multiplier.

    CombMNZ_score(D) = CombSUM_score(D) * n_retrievers_found_D

    The presence multiplier explicitly rewards multi-retriever consensus:
        Retrieved by all 3 retrievers: score is tripled.
        Retrieved by 2 retrievers:     score is doubled.
        Retrieved by 1 retriever only: score is unchanged (1x).

    This is the most commonly cited score-based fusion method that outperforms
    simple CombSUM in IR evaluations (Fox and Shaw, 1994; Bartell et al., 1994).
    The empirical advantage: a weak document retrieved by 3 systems (score 0.4 * 3 = 1.2)
    will outrank a strong document retrieved by 1 system (score 0.9 * 1 = 0.9) when
    the cross-retriever consensus is more informative than score magnitude.
    """
    trace = EnsembleTrace(question=question, strategy="Config3_CombMNZ_PresenceWeighted")
    t0    = time.perf_counter()

    t_ret = time.perf_counter()
    dense_results = retrieve_with_scores(question, vectorstore, config.BASE_K, "similarity")
    bm25_results  = retrieve_bm25_with_scores(question, bm25_base, config.BASE_K)
    mmr_results   = retrieve_with_scores(question, vectorstore, config.BASE_K, "mmr")
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.retriever_contributions = {
        "BGE_Dense": len(dense_results),
        "BM25Plus":  len(bm25_results),
        "MMR_Dense": len(mmr_results),
    }

    retriever_results = [
        ("BGE_Dense", dense_results),
        ("BM25Plus",  bm25_results),
        ("MMR_Dense", mmr_results),
    ]

    t_fuse = time.perf_counter()
    fused_docs, fused_scores, presence = combmnz_fuse(retriever_results, config)
    trace.fusion_ms = (time.perf_counter() - t_fuse) * 1000

    trace.aggregation_method = "CombMNZ (CombSUM * presence multiplier)"
    trace.candidates_total   = len(fused_docs)
    trace.final_docs         = fused_docs[: config.FINAL_K]
    trace.fusion_scores      = {k: v for k, v in list(fused_scores.items())[:20]}
    trace.presence_counts    = presence

    answer, trace.generation_ms = format_context_and_answer(
        question, trace.final_docs, llm, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [15]:

# ---------------------------------------------------------------------------
# CONFIG 4: Borda Count Voting
# ---------------------------------------------------------------------------

def run_config4_borda_count(
    question: str, vectorstore: FAISS, bm25_base: BM25Retriever,
    llm: ChatOllama, config: EnsembleConfig,
) -> EnsembleTrace:
    """
    Configuration 4: Borda Count -- pure rank voting, score-agnostic.

    Borda_score(D) = SUM_{i} max(0, K - rank_0based_i(D))

    At K=10: rank-0 = 10 votes, rank-9 = 1 vote, not in top-10 = 0 votes.
    Linear vote decay (unlike RRF's hyperbolic decay).

    Score-agnostic advantage:
        BM25Plus raw scores (0-20) and BGE cosine similarities (0.7-0.99) are
        never compared directly. Only their rank positions are used.
        This makes Borda Count the correct method when raw scores are not
        available through the retriever API (e.g., when using a third-party
        retrieval service that returns ranked results without scores).

    Borda Count vs RRF (both score-agnostic):
        Borda: linear decay. rank-1 gets K points, rank-K gets 1 point.
               Constant 1-point gap between adjacent ranks.
        RRF:   hyperbolic decay. rank-1 gets 1/(k+1) = 0.016, rank-60 gets 0.0154.
               Smaller gap at higher ranks; more sensitive to rank-1 distinction.
        When to use Borda over RRF:
            You want equal-step discrimination between adjacent ranks.
            You prefer a more interpretable integer vote count over fractional RRF scores.
    """
    trace = EnsembleTrace(question=question, strategy="Config4_BordaCount_RankVoting")
    t0    = time.perf_counter()

    t_ret = time.perf_counter()
    bm25_retriever = BM25Retriever(
        vectorizer=bm25_base.vectorizer, docs=bm25_base.docs,
        k=config.BASE_K, preprocess_func=bm25_base.preprocess_func,
    )
    dense_docs = [d for d, _ in retrieve_with_scores(
        question, vectorstore, config.BASE_K, "similarity")]
    bm25_docs  = bm25_retriever.invoke(question)
    mmr_docs   = [d for d, _ in retrieve_with_scores(
        question, vectorstore, config.BASE_K, "mmr")]
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.retriever_contributions = {
        "BGE_Dense": len(dense_docs),
        "BM25Plus":  len(bm25_docs),
        "MMR_Dense": len(mmr_docs),
    }

    retriever_results = [
        ("BGE_Dense", dense_docs),
        ("BM25Plus",  bm25_docs),
        ("MMR_Dense", mmr_docs),
    ]

    t_fuse = time.perf_counter()
    fused_docs, fused_scores, presence = borda_count_fuse(
        retriever_results, k=config.BORDA_K
    )
    trace.fusion_ms = (time.perf_counter() - t_fuse) * 1000

    trace.aggregation_method = f"Borda Count (rank voting, K={config.BORDA_K})"
    trace.candidates_total   = len(fused_docs)
    trace.final_docs         = fused_docs[: config.FINAL_K]
    trace.fusion_scores      = {k: v for k, v in list(fused_scores.items())[:20]}
    trace.presence_counts    = presence

    answer, trace.generation_ms = format_context_and_answer(
        question, trace.final_docs, llm, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [16]:

# ---------------------------------------------------------------------------
# CONFIG 5: Convex Combination + Dynamic Weight Meta-Retriever [BEST]
# ---------------------------------------------------------------------------

def run_config5_convex_combination_dynamic(
    question: str, vectorstore: FAISS, bm25_base: BM25Retriever,
    llm: ChatOllama, config: EnsembleConfig,
) -> EnsembleTrace:
    """
    Configuration 5: Convex Combination + Dynamic Weight Meta-Retriever [BEST].

    CC_score(D) = SUM_{i} w_i * normalize_i(score_i(D))  with SUM(w_i) = 1.0

    Convex Combination outperforms RRF across all tested corpora with a single
    easy-to-tune parameter (Bruch et al., 2022), especially when training data
    or query-type signals are available to calibrate per-retriever weights.

    The weight vector is ADAPTIVE: it changes per query based on the query router's
    classification of the incoming query as semantic, keyword, or hybrid.

        SEMANTIC query (conceptual, long): w = [0.70, 0.10, 0.20]
            BGE-large dense retrieval gets the highest weight (0.70) because
            it is optimized for semantic similarity and the query is conceptual.
            BM25 is suppressed (0.10) because exact-keyword matching is less
            informative for semantic queries.

        KEYWORD query (technical, short): w = [0.35, 0.50, 0.15]
            BM25Plus gets the highest weight (0.50) because exact-term matching
            is the most reliable signal for queries with specific terminology.
            BGE-large still contributes (0.35) for contextual disambiguation.

        HYBRID query (balanced): w = [0.40, 0.35, 0.25]
            Balanced weights. Both semantic and keyword signals contribute.

    Implementation note on LangChain's EnsembleRetriever vs manual CC:
        LangChain's EnsembleRetriever implements: score = w * (1 / (c + rank))
        -- a WEIGHTED RRF variant, NOT Convex Combination.
        CC requires raw normalized scores, not rank-based scores.
        This configuration uses the manual retrieve_with_scores() and
        convex_combination_fuse() to implement true Convex Combination.

    Args:
        question    : User query.
        vectorstore : FAISS + BGE-large index.
        bm25_base   : BM25Plus retriever.
        llm         : ChatOllama.
        config      : EnsembleConfig.

    Returns:
        EnsembleTrace with query_class, weights_used, and CC fusion scores.
    """
    trace = EnsembleTrace(
        question=question,
        strategy="Config5_ConvexCombination_DynamicWeight [BEST]",
    )
    t0 = time.perf_counter()

    # Route query to determine adaptive weights
    query_class, weights = route_query_class(question)
    trace.query_class = query_class
    trace.weights_used = weights

    logger.info(
        "Config5 CC: query_class='%s' -> weights BGE=%.2f BM25=%.2f MMR=%.2f",
        query_class, weights[0], weights[1], weights[2],
    )

    # Retrieve from all 3 retrievers with scores
    t_ret = time.perf_counter()
    dense_results = retrieve_with_scores(question, vectorstore, config.BASE_K, "similarity")
    bm25_results  = retrieve_bm25_with_scores(question, bm25_base, config.BASE_K)
    mmr_results   = retrieve_with_scores(question, vectorstore, config.BASE_K, "mmr")
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.retriever_contributions = {
        "BGE_Dense": len(dense_results),
        "BM25Plus":  len(bm25_results),
        "MMR_Dense": len(mmr_results),
    }

    retriever_results = [
        ("BGE_Dense", dense_results),
        ("BM25Plus",  bm25_results),
        ("MMR_Dense", mmr_results),
    ]

    # Apply Convex Combination with adaptive weights
    t_fuse = time.perf_counter()
    fused_docs, fused_scores, presence = convex_combination_fuse(
        retriever_results, weights=weights, config=config
    )
    trace.fusion_ms = (time.perf_counter() - t_fuse) * 1000

    trace.aggregation_method = (
        f"Convex Combination (class={query_class}, "
        f"w=[{weights[0]:.2f},{weights[1]:.2f},{weights[2]:.2f}])"
    )
    trace.candidates_total = len(fused_docs)
    trace.final_docs       = fused_docs[: config.FINAL_K]
    trace.fusion_scores    = {k: v for k, v in list(fused_scores.items())[:20]}
    trace.presence_counts  = presence

    answer, trace.generation_ms = format_context_and_answer(
        question, trace.final_docs, llm, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [17]:


# ===========================================================================
# SECTION 7 -- FUSION METHOD COMPARISON UTILITY
# ===========================================================================

def compare_fusion_methods(
    question: str, vectorstore: FAISS, bm25_base: BM25Retriever,
    config: EnsembleConfig,
) -> None:
    """
    Utility to compare CombSUM, CombMNZ, Borda Count, and CC rankings
    side-by-side for a single query, without LLM answer generation.

    Useful for calibrating fusion method selection on your labeled query set.
    Run this on 20-50 labeled queries to determine which fusion method
    most often places ground-truth relevant documents in the top-4 positions.

    Each column shows the dedup_key of the top-4 documents under each method.
    Differences between columns reveal the practical impact of method choice.
    """
    dense_results = retrieve_with_scores(question, vectorstore, config.BASE_K, "similarity")
    bm25_results  = retrieve_bm25_with_scores(question, bm25_base, config.BASE_K)
    mmr_results   = retrieve_with_scores(question, vectorstore, config.BASE_K, "mmr")

    retriever_results_scored = [
        ("BGE_Dense", dense_results),
        ("BM25Plus",  bm25_results),
        ("MMR_Dense", mmr_results),
    ]
    retriever_results_ranked = [
        ("BGE_Dense", [d for d, _ in dense_results]),
        ("BM25Plus",  [d for d, _ in bm25_results]),
        ("MMR_Dense", [d for d, _ in mmr_results]),
    ]
    qclass, weights = route_query_class(question)

    csum_docs,  _, _ = combsum_fuse(retriever_results_scored, config)
    cmnz_docs,  _, _ = combmnz_fuse(retriever_results_scored, config)
    borda_docs, _, _ = borda_count_fuse(retriever_results_ranked, config.BORDA_K)
    cc_docs,    _, _ = convex_combination_fuse(retriever_results_scored, weights, config)

    def label(doc: Document) -> str:
        return (f"{doc.metadata.get('paper_name','?')[:15]} "
                f"p{doc.metadata.get('page','?')}")

    print(f"\n{'='*80}")
    print(f"FUSION METHOD COMPARISON | query_class={qclass}")
    print(f"Query: {question[:60]}")
    print(f"{'Rank':<6} {'CombSUM':<22} {'CombMNZ':<22} {'Borda':<22} {'CC(adaptive)':<22}")
    print("-" * 80)
    for i in range(config.FINAL_K):
        row = [f"#{i+1}"]
        for docs in [csum_docs, cmnz_docs, borda_docs, cc_docs]:
            row.append(label(docs[i]) if i < len(docs) else "---")
        print(f"{row[0]:<6} {row[1]:<22} {row[2]:<22} {row[3]:<22} {row[4]:<22}")
    print("=" * 80 + "\n")



In [18]:


# ===========================================================================
# SECTION 8 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question: str, vectorstore: FAISS, bm25_base: BM25Retriever,
    llm: ChatOllama, config: EnsembleConfig,
) -> Dict[str, EnsembleTrace]:
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline":               lambda q: run_config1_baseline(
            q, vectorstore, llm, config),
        "Config2_CombSUM":                lambda q: run_config2_combsum(
            q, vectorstore, bm25_base, llm, config),
        "Config3_CombMNZ":                lambda q: run_config3_combmnz(
            q, vectorstore, bm25_base, llm, config),
        "Config4_BordaCount":             lambda q: run_config4_borda_count(
            q, vectorstore, bm25_base, llm, config),
        "Config5_CC_DynamicWeight [BEST]": lambda q: run_config5_convex_combination_dynamic(
            q, vectorstore, bm25_base, llm, config),
    }

    traces: Dict[str, EnsembleTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("ENSEMBLE RAG AGGREGATION SUMMARY")
    print(f"{'Config':<46} {'Cands':>6} {'Final':>6} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<46} {tr.candidates_total:>6} "
            f"{len(tr.final_docs):>6} {tr.total_ms:>10.0f}"
        )
    print("=" * 78 + "\n")

    return traces



In [19]:
"""
    End-to-end Ensemble RAG pipeline orchestrator.

    LLM calls per query: 1 (answer generation only) for all 5 configs.
    Zero LLM calls for fusion -- all aggregation methods run locally.

    Fusion method latency (excluding retrieval and LLM):
        CombSUM:  < 5ms  (normalization + summation over ~30 candidates)
        CombMNZ:  < 5ms  (CombSUM + integer multiply)
        Borda:    < 5ms  (rank-position voting over ~30 candidates)
        CC:       < 6ms  (normalization + weighted summation + route_query_class < 1ms)
    """

'\n    End-to-end Ensemble RAG pipeline orchestrator.\n\n    LLM calls per query: 1 (answer generation only) for all 5 configs.\n    Zero LLM calls for fusion -- all aggregation methods run locally.\n\n    Fusion method latency (excluding retrieval and LLM):\n        CombSUM:  < 5ms  (normalization + summation over ~30 candidates)\n        CombMNZ:  < 5ms  (CombSUM + integer multiply)\n        Borda:    < 5ms  (rank-position voting over ~30 candidates)\n        CC:       < 6ms  (normalization + weighted summation + route_query_class < 1ms)\n    '

In [20]:
config = EnsembleConfig()
logger.info("=== Ensemble RAG (Multiple Retrievers Aggregation) Pipeline Starting ===")


2026-05-21 19:28:51 | INFO     | ensemble_rag | === Ensemble RAG (Multiple Retrievers Aggregation) Pipeline Starting ===


In [21]:
pdf_paths   = download_pdfs(config)


2026-05-21 19:29:02 | INFO     | ensemble_rag | Downloading: https://arxiv.org/pdf/1706.03762.pdf
2026-05-21 19:29:04 | INFO     | ensemble_rag | Saved: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-21 19:29:05 | INFO     | ensemble_rag | Downloading: https://arxiv.org/pdf/1810.04805.pdf
2026-05-21 19:29:06 | INFO     | ensemble_rag | Saved: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-21 19:29:07 | INFO     | ensemble_rag | Downloading: https://arxiv.org/pdf/2005.11401.pdf
2026-05-21 19:29:08 | INFO     | ensemble_rag | Saved: rag_knowledge_intensive_nlp.pdf  (864.6 KB)


In [22]:
chunks      = load_and_chunk_documents(pdf_paths, config)


2026-05-21 19:29:27 | INFO     | ensemble_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-21 19:29:27 | INFO     | ensemble_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-21 19:29:28 | INFO     | ensemble_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-21 19:29:28 | INFO     | ensemble_rag | Total chunks: 458


In [23]:
embeddings  = build_bge_embeddings(config)


2026-05-21 19:29:41 | INFO     | ensemble_rag | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-21 19:29:41 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


C:\Users\dhanu\AppData\Local\Temp\ipykernel_29020\3372733953.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-05-21 19:29:42 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-05-21 19:29:42 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-21 19:29:42 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-21 19:29:42 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 19:29:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-21 19:29:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6058.00it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-21 19:29:47 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 19:29:47 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-21 19:29:48 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 19:29:48 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-21 19:29:48 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-21 19:29:49 | INFO     | httpx |

In [24]:
vectorstore = build_faiss_index(chunks, embeddings, config)


2026-05-21 19:30:32 | INFO     | ensemble_rag | Building FAISS from 458 chunks ...
2026-05-21 19:33:30 | INFO     | faiss.loader | Loading faiss with AVX512 support.
2026-05-21 19:33:30 | INFO     | faiss.loader | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-05-21 19:33:30 | INFO     | faiss.loader | Loading faiss with AVX2 support.
2026-05-21 19:33:30 | INFO     | faiss.loader | Successfully loaded faiss with AVX2 support.


In [32]:
bm25_base   = build_bm25_index(chunks, config)


2026-05-21 19:37:26 | WARNING  | ensemble_rag | BM25 cache unreadable/corrupted; rebuilding cache.
2026-05-21 19:37:26 | WARNING  | ensemble_rag | BM25Plus init not supported by installed dependencies; falling back to BM25Okapi-compatible params.


In [33]:
llm         = build_ollama_llm(config)


In [34]:

demo_questions = [
    # Exact-keyword -- BM25 should dominate; CC routes to keyword weights
    "What BLEU score did the Transformer base model achieve on WMT 2014 English-to-German?",

    # Semantic conceptual -- BGE-dense should dominate; CC routes to semantic weights
    "How does scaled dot-product attention compute its output as a weighted sum of values?",

    # Hybrid multi-aspect -- balanced routing
    "What training procedure and optimizer settings are used in the original Transformer?",

    # Presence consensus test -- all retrievers should agree on core topic
    "What is the architecture of the Transformer encoder?",

    # CombMNZ consensus advantage test -- multi-hop query
    "How do BERT and the original Transformer differ in how they use pre-training?",
]


In [35]:
# Run fusion method comparison utility (no LLM, for calibration)
logger.info("Running fusion method comparison (no LLM calls) ...")
for q in demo_questions[:2]:
    compare_fusion_methods(q, vectorstore, bm25_base, config)

# Run all 5 configs on all demo questions
for question in demo_questions:
    run_all_configs(question, vectorstore, bm25_base, llm, config)

logger.info("=== Ensemble RAG Pipeline Demo Complete ===")


2026-05-21 19:37:39 | INFO     | ensemble_rag | Running fusion method comparison (no LLM calls) ...

FUSION METHOD COMPARISON | query_class=keyword
Query: What BLEU score did the Transformer base model achieve on WM
Rank   CombSUM                CombMNZ                Borda                  CC(adaptive)          
--------------------------------------------------------------------------------
#1     Attention Is Al p9     Attention Is Al p9     Attention Is Al p7     Attention Is Al p9    
#2     Attention Is Al p0     Attention Is Al p0     Attention Is Al p7     Attention Is Al p0    
#3     Attention Is Al p8     Attention Is Al p8     Attention Is Al p0     Attention Is Al p7    
#4     Attention Is Al p8     Attention Is Al p8     Attention Is Al p8     Attention Is Al p0    


FUSION METHOD COMPARISON | query_class=semantic
Query: How does scaled dot-product attention compute its output as 
Rank   CombSUM                CombMNZ                Borda                  CC(adaptive)  